# Publications markdown generator for Sangeet

Scrapes ORCID, builds a table of publications with metadata and converts them for use with [sangeetpaul.github.io](sangeetpaul.github.io).

In [1]:
import os
import requests
import pandas as pd

In [2]:
orcid = '0000-0002-4449-1732'

In [3]:
response = requests.get('https://pub.orcid.org/v3.0/{}/works'.format(orcid),
                        headers={"Accept": "application/orcid+json" })
record = response.json()

In [4]:
print(f"Number of works = {len(record['group'])}")

Number of works = 63


In [5]:
put_codes = []
for work in record['group']:
    put_code = work['work-summary'][0]['put-code']
    put_codes.append(put_code)
put_code = put_codes[0]

In [6]:
works = []
for put_code in put_codes:
    response = requests.get('https://pub.orcid.org/v3.0/{}/work/{}'.format(orcid, put_code),
                            headers={"Accept": "application/orcid+json" })
    work = response.json()
    works.append(work)
work = works[0]

In [7]:
pub_list = []
for work in works:
    pub_date = work['publication-date']['year']['value']+"-"+work['publication-date']['month']['value']+"-01"
    pub_cite = "LVK "+work['publication-date']['year']['value']+", "+work['title']['title']['value']+", "+work['journal-title']['value']
    for ext_id in work['external-ids']['external-id']:
        if ext_id['external-id-type'] == 'doi':
            pub_url = "https://doi.org/"+ext_id['external-id-value']
            break
    pub_data = [
        pub_date,
        work['title']['title']['value'],
        work['journal-title']['value'],
        None,
        pub_cite,
        str(work['put-code']),
        pub_url,
        None,
        "manuscripts",
    ]
    pub_list.append(pub_data)

In [8]:
publications = pd.DataFrame(pub_list, columns=['pub_date','title','venue','excerpt','citation','url_slug','paper_url','slides_url','category'])
publications

,pub_date,title,venue,excerpt,citation,url_slug,paper_url,slides_url,category
0,2026-03-01,All-sky Searches for Continuous Gravitational ...,arXiv e-prints,None,"LVK 2026, All-sky Searches for Continuous Grav...",210034188,https://doi.org/10.48550/arXiv.2603.14168,None,manuscripts
1,2026-03-01,GWTC-4.0: Tests of General Relativity. I. Over...,arXiv e-prints,None,"LVK 2026, GWTC-4.0: Tests of General Relativit...",210034185,https://doi.org/10.48550/arXiv.2603.19019,None,manuscripts
2,2026-03-01,GWTC-4.0: Tests of General Relativity. II. Par...,arXiv e-prints,None,"LVK 2026, GWTC-4.0: Tests of General Relativit...",210034184,https://doi.org/10.48550/arXiv.2603.19020,None,manuscripts
3,2026-03-01,GWTC-4.0: Tests of General Relativity. III. Te...,arXiv e-prints,None,"LVK 2026, GWTC-4.0: Tests of General Relativit...",210034103,https://doi.org/10.48550/arXiv.2603.19021,None,manuscripts
4,2026-01-01,Black Hole Spectroscopy and Tests of General R...,Physical Review Letters,None,"LVK 2026, Black Hole Spectroscopy and Tests of...",210034191,https://doi.org/10.1103/6c61-fm1n,None,manuscripts
...,...,...,...,...,...,...,...,...,...
58,2022-03-01,Search for intermediate-mass black hole binari...,Astronomy and Astrophysics,None,"LVK 2022, Search for intermediate-mass black h...",182162574,https://doi.org/10.1051/0004-6361/202141452,None,manuscripts
59,2022-01-01,Search for continuous gravitational waves from...,Physical Review D,None,"LVK 2022, Search for continuous gravitational ...",182162575,https://doi.org/10.1103/PhysRevD.105.022002,None,manuscripts
60,2021-12-01,All-sky search for short gravitational-wave bu...,Physical Review D,None,"LVK 2021, All-sky search for short gravitation...",182162605,https://doi.org/10.1103/PhysRevD.104.122004,None,manuscripts
61,2021-12-01,Tests of General Relativity with GWTC-3,arXiv e-prints,None,"LVK 2021, Tests of General Relativity with GWT...",182162604,https://doi.org/10.48550/arXiv.2112.06861,None,manuscripts


In [9]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

In [10]:
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""

    md += """\ncategory: manuscripts"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.slides_url)) > 5:
        md += "\nslidesurl: '" + item.slides_url + "'"

    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"

    if len(str(item.slides_url)) > 5:
        md += "\n[Download slides here](" + item.slides_url + ")\n" 

    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)